In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display


def resolve_results_dir() -> Path:
    """Prefer ./results when the kernel cwd is `notebooks/`; also try repo-relative paths."""
    for candidate in (
        Path("results"),
        Path("notebooks/results"),
        Path.cwd() / "results",
    ):
        if candidate.exists() and candidate.is_dir():
            return candidate.resolve()
    raise FileNotFoundError(
        "Could not find a results folder. Run this notebook with cwd = `notebooks/` "
        "or create `./results` next to this notebook."
    )


RESULTS = resolve_results_dir()
print(f"Results directory: {RESULTS}\n")

csv_files = sorted(RESULTS.rglob("*.csv"))
print(f"Found {len(csv_files)} CSV file(s).\n")

merics_parts: list[pd.DataFrame] = []
trade_inventory: list[dict] = []

for fp in csv_files:
    rel = fp.relative_to(RESULTS)
    parts = rel.parts
    strategy = symbol = timeframe = ""
    if len(parts) >= 4:
        strategy, symbol, timeframe = parts[0], parts[1], parts[2]
    elif len(parts) >= 2:
        strategy = parts[0]

    try:
        df = pd.read_csv(fp)
    except Exception as exc:
        print(f"[skip] {rel} — {exc}")
        continue

    low = fp.name.lower()
    if low == "merics.csv":
        extra = pd.DataFrame(
            {
                "strategy": [strategy] * len(df),
                "symbol": [symbol] * len(df),
                "timeframe": [timeframe] * len(df),
                "rel_path": [str(rel)] * len(df),
            }
        )
        merics_parts.append(pd.concat([extra.reset_index(drop=True), df.reset_index(drop=True)], axis=1))
    elif low == "trades.csv":
        trade_inventory.append(
            {
                "strategy": strategy,
                "symbol": symbol,
                "timeframe": timeframe,
                "rows": len(df),
                "rel_path": str(rel),
            }
        )

display(Markdown("### Combined metrics (`merics.csv`)"))
if merics_parts:
    all_metrics = pd.concat(merics_parts, ignore_index=True)
    front = ["strategy", "symbol", "timeframe", "rel_path"]
    rest = [c for c in all_metrics.columns if c not in front]
    all_metrics = all_metrics[front + sorted(rest, key=str.lower)]
    display(
        all_metrics.sort_values(["strategy", "symbol", "timeframe"]).reset_index(drop=True)
    )
else:
    display(Markdown("_No `merics.csv` files found._"))

display(Markdown("### Trades files (`trades.csv`)"))
if trade_inventory:
    display(
        pd.DataFrame(trade_inventory).sort_values(
            ["strategy", "symbol", "timeframe"]
        ).reset_index(drop=True)
    )
else:
    display(Markdown("_No `trades.csv` files found._"))

display(Markdown("### Every CSV path (relative to results root)"))
for fp in csv_files:
    print(fp.relative_to(RESULTS))

Results directory: D:\bot\ema-1d trend\notebooks\results

Found 64 CSV file(s).



### Combined metrics (`merics.csv`)

,strategy,symbol,timeframe,rel_path,avg_pnl,end_balance,max_drawdown_%,net_pnl,profit_factor,return_%,start_balance,trades,win_rate_%
0,strategy03,AUDCAD,M5,strategy03\AUDCAD\M5\merics.csv,2.64,12800.61,20.92,2800.61,1.046,28.01,10000.0,1062,51.41
1,strategy03,AUDCHF,M5,strategy03\AUDCHF\M5\merics.csv,-2.89,7451.57,33.76,-2548.43,0.940,-25.48,10000.0,883,48.58
2,strategy03,AUDJPY,M5,strategy03\AUDJPY\M5\merics.csv,13.78,27373.29,13.83,17373.29,1.159,173.73,10000.0,1261,54.24
3,strategy03,AUDNZD,M5,strategy03\AUDNZD\M5\merics.csv,-5.02,5756.42,46.06,-4243.58,0.868,-42.44,10000.0,845,46.98
4,strategy03,AUDUSD,M5,strategy03\AUDUSD\M5\merics.csv,1.79,11927.42,22.67,1927.42,1.033,19.27,10000.0,1075,51.07
5,strategy03,CADCHF,M5,strategy03\CADCHF\M5\merics.csv,-4.30,7090.27,50.43,-2909.73,0.921,-29.10,10000.0,677,47.71
6,strategy03,CADJPY,M5,strategy03\CADJPY\M5\merics.csv,15.54,29919.80,17.02,19919.80,1.206,199.20,10000.0,1282,54.52
7,strategy03,EURCZK,M5,strategy03\EURCZK\M5\merics.csv,-1.96,6134.43,65.69,-3865.57,0.955,-38.66,10000.0,1973,49.01
8,strategy03,EURHUF,M5,strategy03\EURHUF\M5\merics.csv,11.46,33165.94,79.12,23165.94,1.193,231.66,10000.0,2022,53.21
9,strategy03,EURTRY,M5,strategy03\EURTRY\M5\merics.csv,6.34,23683.11,42.76,13683.11,1.088,136.83,10000.0,2157,52.25


### Trades files (`trades.csv`)

,strategy,symbol,timeframe,rows,rel_path
0,strategy03,AUDCAD,M5,1062,strategy03\AUDCAD\M5\trades.csv
1,strategy03,AUDCHF,M5,883,strategy03\AUDCHF\M5\trades.csv
2,strategy03,AUDJPY,M5,1261,strategy03\AUDJPY\M5\trades.csv
3,strategy03,AUDNZD,M5,845,strategy03\AUDNZD\M5\trades.csv
4,strategy03,AUDUSD,M5,1075,strategy03\AUDUSD\M5\trades.csv
5,strategy03,CADCHF,M5,677,strategy03\CADCHF\M5\trades.csv
6,strategy03,CADJPY,M5,1282,strategy03\CADJPY\M5\trades.csv
7,strategy03,EURCZK,M5,1973,strategy03\EURCZK\M5\trades.csv
8,strategy03,EURHUF,M5,2022,strategy03\EURHUF\M5\trades.csv
9,strategy03,EURTRY,M5,2157,strategy03\EURTRY\M5\trades.csv


### Every CSV path (relative to results root)

strategy03\AUDCAD\M5\merics.csv
strategy03\AUDCAD\M5\trades.csv
strategy03\AUDCHF\M5\merics.csv
strategy03\AUDCHF\M5\trades.csv
strategy03\AUDJPY\M5\merics.csv
strategy03\AUDJPY\M5\trades.csv
strategy03\AUDNZD\M5\merics.csv
strategy03\AUDNZD\M5\trades.csv
strategy03\AUDUSD\M5\merics.csv
strategy03\AUDUSD\M5\trades.csv
strategy03\CADCHF\M5\merics.csv
strategy03\CADCHF\M5\trades.csv
strategy03\CADJPY\M5\merics.csv
strategy03\CADJPY\M5\trades.csv
strategy03\EURCZK\M5\merics.csv
strategy03\EURCZK\M5\trades.csv
strategy03\EURHUF\M5\merics.csv
strategy03\EURHUF\M5\trades.csv
strategy03\EURTRY\M5\merics.csv
strategy03\EURTRY\M5\trades.csv
strategy03\EURUSD\M5\merics.csv
strategy03\EURUSD\M5\trades.csv
strategy03\EURZAR\M5\merics.csv
strategy03\EURZAR\M5\trades.csv
strategy03\GBPTRY\M5\merics.csv
strategy03\GBPTRY\M5\trades.csv
strategy03\GBPZAR\M5\merics.csv
strategy03\GBPZAR\M5\trades.csv
strategy03\NZDJPY\M5\merics.csv
strategy03\NZDJPY\M5\trades.csv
strategy03\NZDUSD\M5\merics.csv
strategy